In [3]:
import os
import sys
from datetime import datetime
import itertools

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(42)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:

# Create one timestamp for the entire KAN experiment batch
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

results_dir = f"results_accuracy_efficiency_{timestamp}"

print(f"Results will be saved to: {results_dir}")


# Optimal hyperparameters found in 02_hyperparameter_tunning (results_kan_infinite_optuna_2026-09-20_01-43-35)
optimal_grid_size = 5
optimal_spline_order = 4
optimal_adam_lr = 1e-4

# Configurations to sweep: L (hidden layers) x width (hidden units), keeping the rest of the hyperparameters optimal
L_values = [1, 2, 3]
width_values = [15, 25, 35]
kan_configurations = [
    {"L": L, "width": width, "grid_size": optimal_grid_size, "spline_order": optimal_spline_order}
    for L in L_values
    for width in width_values
]


# Loop through each table configuration
for cfg in kan_configurations:

    layers = cfg["L"]
    width = cfg["width"]
    grid_sz = cfg["grid_size"]
    spline_ord = cfg["spline_order"]

    print(
        f"\n--- Running KAN Experiment: "
        f"Layers (L)={layers}, Width (N)={width}, Grid={grid_sz}, Spline Order={spline_ord} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="KAN",
            hidden_layers=layers,
            hidden_units=width,
            grid_size=grid_sz,
            spline_order=spline_ord,
            adam_lr=optimal_adam_lr,
            device=device,
            adam_iters=2000,
            lbfgs_iters=2000,
            results_dir=results_dir,
        )

        print(
            f"Success! Time: {compute_time:.2f}s | "
            f"Err U: {err_u:.3e} | Err K: {err_k:.3e}"
        )

    except Exception as e:
        print(
            f"Experiment failed for "
            f"Layers={layers}, Width={width}, Grid={grid_sz} "
            f"with error: {e}"
        )

Results will be saved to: results_accuracy_efficiency_2026-09-20_14-13-31

--- Running KAN Experiment: Layers (L)=1, Width (N)=15, Grid=5, Spline Order=4 ---

[KAN] L=1, N=15 | Params: 990 | Mean Err: 2.879e-01 | Saved to 'results_accuracy_efficiency_2026-09-20_14-13-31/'.
Success! Time: 424.65s | Err U: 2.941e-02 | Err K: 5.463e-01

--- Running KAN Experiment: Layers (L)=1, Width (N)=25, Grid=5, Spline Order=4 ---

[KAN] L=1, N=25 | Params: 1,650 | Mean Err: 6.890e-02 | Saved to 'results_accuracy_efficiency_2026-09-20_14-13-31/'.
Success! Time: 375.00s | Err U: 3.050e-02 | Err K: 1.073e-01

--- Running KAN Experiment: Layers (L)=1, Width (N)=35, Grid=5, Spline Order=4 ---

[KAN] L=1, N=35 | Params: 2,310 | Mean Err: 1.793e-02 | Saved to 'results_accuracy_efficiency_2026-09-20_14-13-31/'.
Success! Time: 269.95s | Err U: 2.495e-02 | Err K: 1.090e-02

--- Running KAN Experiment: Layers (L)=2, Width (N)=15, Grid=5, Spline Order=4 ---

[KAN] L=2, N=15 | Params: 5,940 | Mean Err: 6.492e-03 